<a href="https://colab.research.google.com/github/digorms/tcc-ia-expansao-urbana/blob/main/notebooks/Modelo_de_Escolha_de_Novas_Igrejas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

#from google.colab import drive
#drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Carregar o CSV Final exportado do QGIS
# O pandas vai ler usando o separador ponto e vírgula ';' conforme configurado
#df = pd.read_csv('/content/drive/MyDrive/TCC/RJ_Bairros.csv', sep=';')
url_github = 'https://raw.githubusercontent.com/digorms/tcc-ia-expansao-urbana/refs/heads/main/data/RJ_Bairros.csv'
df = pd.read_csv(url_github, sep=';')

# 2. Como o CSV original traz dados de outros municípios, vamos filtrar apenas para a capital
# Garanta que a coluna NM_MUN ou similar filtre apenas o Rio de Janeiro
df_rj = df[df['NM_MUN'] == 'Rio de Janeiro'].copy()

print(df_rj[['NM_MUN', 'NM_BAIRRO', 'RB_REND_ME', 'Num_Igreja']].head(5))
print("\n--- Tipos de Dados ---")
print(df_rj[['RB_REND_ME', 'Num_metro', 'Num_BRT', 'Num_Pontos', 'Num_Igreja']].dtypes)

# 3. Definir as Variáveis Preditores (Features) e a Variável Alvo (Target)
features = ['RB_REND_ME', 'Num_metro', 'Num_BRT', 'Num_Pontos']
target = 'Num_Igreja'

X = df_rj[features]
y = df_rj[target]

# 4. Tratamento preventivo de segurança (substituir vazios por zero)
X = X.fillna(0)
y = y.fillna(0)

# 5. Divisão Estatística (80% Treino, 20% Teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Inicializar e Treinar o RandomForest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 7. Avaliação do Modelo nos dados de Teste
y_pred = model.predict(X_test)

print("=== RELATÓRIO DE PERFORMANCE DA IA ===")
print(f"R² Score (Poder de Explicação): {r2_score(y_test, y_pred):.2f}")
print(f"Erro Médio Absoluto (MAE): {mean_absolute_error(y_test, y_pred):.2f} igrejas")

# 8. ENGENHARIA DE DECISÃO: Mapeando as Oportunidades
# Calculamos a previsão para TODOS os bairros do Rio de Janeiro
df_rj['Igrejas_Preditas'] = model.predict(X)

# 9. Criamos o indicador de Oportunidade / Saturação
df_rj['Oportunidade_Expansao'] = df_rj['Igrejas_Preditas'] - df_rj['Num_Igreja']

# 10. Exibir os 10 bairros com maior potencial de expansão (onde faltam igrejas segundo o modelo)
ranking_oportunidades = df_rj[['NM_BAIRRO', 'RB_REND_ME', 'Num_Igreja', 'Igrejas_Preditas', 'Oportunidade_Expansao']]
print("\n=== TOP 10 BAIRROS COM MAIOR POTENCIAL DE EXPANSÃO ===")
print(ranking_oportunidades.sort_values(by='Oportunidade_Expansao', ascending=False).head(10).to_string(index=False))